In [8]:
import os
import subprocess
import time

# 1. LOCAL PROJECT CONFIGURATION 
# Paths updated to match your Weather & Energy project structure
PROJECT_ROOT = r"C:\Users\abeer\Desktop\weather_energy_pipeline_"
BRONZE_PATH  = os.path.join(PROJECT_ROOT, "data", "bronze")
GOLD_PATH    = os.path.join(PROJECT_ROOT, "data", "gold")

print(f" Project root  : {PROJECT_ROOT}")
print(f" Bronze Path   : {BRONZE_PATH} (Exists: {os.path.exists(BRONZE_PATH)})")
print(f" Gold Path     : {GOLD_PATH} (Exists: {os.path.exists(GOLD_PATH)})")

 Project root  : C:\Users\abeer\Desktop\weather_energy_pipeline_
 Bronze Path   : C:\Users\abeer\Desktop\weather_energy_pipeline_\data\bronze (Exists: True)
 Gold Path     : C:\Users\abeer\Desktop\weather_energy_pipeline_\data\gold (Exists: True)


In [30]:
# AWS Deployment Configuration
AWS_REGION      = "eu-north-1"
S3_BUCKET       = "weather-energy-b5data"
S3_PATH         = f"s3://{S3_BUCKET}/data/processed/flights_clean.parquet"
EC2_INSTANCE_ID = "i-0704c4b2195d8e645"  # From your deployment guide
IAM_ROLE        = "ec2-s3-weather-role"
EC2_IP          = "13.51.204.144"

print("=" * 55)
print("AWS DEPLOYMENT CONFIGURATION")
print("=" * 55)
print(f" Region       : {AWS_REGION}")
print(f" S3 Bucket    : {S3_BUCKET}")
print(f" S3 Path      : {S3_PATH}")
print(f" EC2 Instance : {EC2_INSTANCE_ID}")
print(f" EC2 IP       : {EC2_IP}")
print(f" IAM Role     : {IAM_ROLE}")
print("=" * 55)

AWS DEPLOYMENT CONFIGURATION
 Region       : eu-north-1
 S3 Bucket    : weather-energy-b5data
 S3 Path      : s3://weather-energy-b5data/data/processed/flights_clean.parquet
 EC2 Instance : i-0704c4b2195d8e645
 EC2 IP       : 13.51.204.144
 IAM Role     : ec2-s3-weather-role


In [31]:
import subprocess

result = subprocess.run("aws --version", capture_output=True, text=True, shell=True)
print(f" AWS CLI: {result.stdout.strip()}")

result2 = subprocess.run("aws s3 ls", capture_output=True, text=True, shell=True)
print(f" S3 Buckets:\n{result2.stdout.strip()}")

 AWS CLI: aws-cli/1.44.63 Python/3.12.7 Windows/11 botocore/1.42.73
 S3 Buckets:



In [32]:
import os

# Set this to the folder where your project is located
PROJECT_ROOT = r"C:\Users\abeer\Desktop\weather_energy_pipeline_"

# Define the paths to your data folders
BRONZE_PATH = os.path.join(PROJECT_ROOT, "data", "bronze")
GOLD_PATH   = os.path.join(PROJECT_ROOT, "data", "gold")
AWS_REGION  = "eu-north-1"
S3_BUCKET   = "weather-energy-b5data"

print(f"Bronze Path exists: {os.path.exists(BRONZE_PATH)}")
print(f"Gold Path exists: {os.path.exists(GOLD_PATH)}")

Bronze Path exists: True
Gold Path exists: True


In [33]:
import subprocess
import time

print("=" * 55)
print("UPLOADING DATA TO AWS S3...")
print("=" * 55)

start = time.time()

# Upload Bronze Data
cmd_bronze = f'aws s3 sync "{BRONZE_PATH}" s3://{S3_BUCKET}/bronze --region {AWS_REGION}'
subprocess.run(cmd_bronze, shell=True)

# Upload Gold Data
cmd_gold = f'aws s3 sync "{GOLD_PATH}" s3://{S3_BUCKET}/gold --region {AWS_REGION}'
subprocess.run(cmd_gold, shell=True)

duration = round(time.time() - start, 2)
print(f"\nUpload complete in {duration} seconds!")

UPLOADING DATA TO AWS S3...

Upload complete in 5.82 seconds!


In [34]:
# 3. UPLOAD DATA TO AWS S3 
print("\nUPLOADING DATA TO AWS S3...")
start = time.time()

# Sync Bronze Data
cmd_bronze = f'aws s3 sync "{BRONZE_PATH}" s3://{S3_BUCKET}/bronze --region {AWS_REGION}'
subprocess.run(cmd_bronze, shell=True)

# Sync Gold Data
cmd_gold = f'aws s3 sync "{GOLD_PATH}" s3://{S3_BUCKET}/gold --region {AWS_REGION}'
subprocess.run(cmd_gold, shell=True)

duration = round(time.time() - start, 2)
print(f"\nUpload complete in {duration} seconds!")


UPLOADING DATA TO AWS S3...

Upload complete in 6.1 seconds!


In [35]:
# 4. VERIFY S3 CONTENT [cite: 292]
print("\nVERIFYING S3 CONTENT:")
subprocess.run(f"aws s3 ls s3://{S3_BUCKET}/ --recursive --human-readable --summarize", shell=True)


VERIFYING S3 CONTENT:


CompletedProcess(args='aws s3 ls s3://weather-energy-b5data/ --recursive --human-readable --summarize', returncode=255)

In [36]:
# 5. EC2 EXECUTION COMMAND [cite: 206]
# This is the specialized command to run inside your EC2 browser terminal
ec2_command = """
python3 -c "import pyarrow.parquet as pq,pandas as pd,os,time; \\
start=time.time(); \\
files=[os.path.join(r,f) for r,d,fs in os.walk(os.path.expanduser('~/weather_energy_pipeline/data/gold/')) for f in fs if f.endswith('.parquet')]; \\
df=pd.concat([pq.read_table(f).to_pandas() for f in files],ignore_index=True); \\
print('WEATHER ENERGY PIPELINE ON AWS EC2'); \\
print('===================================='); \\
print(f'Total records: {len(df):,}'); \\
corr=df['avg_temp'].corr(df['avg_spot_price']); \\
print(f'Temp vs Price correlation: {corr:.4f}'); \\
print(f'Time: {round(time.time()-start,2)}s'); \\
print('Done! Region:eu-north-1 S3:weather-energy-b5data')"
"""

print("\nCOPY AND RUN THIS ON YOUR EC2 TERMINAL:")
print("=" * 55)
print(ec2_command)
print("=" * 55)


COPY AND RUN THIS ON YOUR EC2 TERMINAL:

python3 -c "import pyarrow.parquet as pq,pandas as pd,os,time; \
start=time.time(); \
files=[os.path.join(r,f) for r,d,fs in os.walk(os.path.expanduser('~/weather_energy_pipeline/data/gold/')) for f in fs if f.endswith('.parquet')]; \
df=pd.concat([pq.read_table(f).to_pandas() for f in files],ignore_index=True); \
print('WEATHER ENERGY PIPELINE ON AWS EC2'); \
print('===================================='); \
print(f'Total records: {len(df):,}'); \
corr=df['avg_temp'].corr(df['avg_spot_price']); \
print(f'Temp vs Price correlation: {corr:.4f}'); \
print(f'Time: {round(time.time()-start,2)}s'); \
print('Done! Region:eu-north-1 S3:weather-energy-b5data')"



In [37]:
# 2. AWS CLOUD CONFIGURATION (Ensure these match your variables)
AWS_REGION      = "eu-north-1"
S3_BUCKET       = "weather-energy-b5data"
EC2_INSTANCE_ID = "i-0704c4b2195d8e645"
EC2_IP          = "13.51.204.144" # Update this if your Public IP changed

# ──────────────────────────────────────────────────────────
# EC2 COMMANDS FOR WEATHER & ENERGY PROJECT
# ──────────────────────────────────────────────────────────

EC2_COMMANDS = f"""
# ── Step 1: Update system ─────────────────────────
sudo yum update -y

# ── Step 2: Install pip ───────────────────────────
sudo yum install python3-pip -y

# ── Step 3: Install Python libraries ─────────────
pip3 install pandas pyarrow boto3 matplotlib seaborn

# ── Step 4: Create project folder ────────────────
mkdir -p ~/weather_energy_pipeline/data/gold

# ── Step 5: Download data from S3 to EC2 ─────────
aws s3 sync s3://{S3_BUCKET}/gold/ ~/weather_energy_pipeline/data/gold/ --region {AWS_REGION}

# ── Step 6: Run Weather & Energy Pipeline Analysis ──
python3 -c "
import pyarrow.parquet as pq, pandas as pd, os, time
start = time.time()
# Find all parquet files in the gold directory
files = [os.path.join(r, f) for r, d, fs in os.walk(os.path.expanduser('~/weather_energy_pipeline/data/gold/')) for f in fs if f.endswith('.parquet')]
# Load and combine data
df = pd.concat([pq.read_table(f).to_pandas() for f in files], ignore_index=True)

print('WEATHER ENERGY PIPELINE ON AWS EC2')
print('====================================')
print(f'Total records: {{len(df):,}}')
# Correlation analysis specific to your project
corr = df['avg_temp'].corr(df['avg_spot_price'])
print(f'Temp vs Price correlation: {{corr:.4f}}')
print(f'Execution Time: {{round(time.time()-start,2)}}s')
print('Done! Region:{AWS_REGION} S3:{S3_BUCKET}')
"
"""

print("=" * 55)
print("EC2 COMMANDS — Copy and run these in EC2 browser terminal")
print("=" * 55)
print(EC2_COMMANDS)
print("=" * 55)
print(f"EC2 Instance : {EC2_INSTANCE_ID}")
print(f"EC2 IP       : {EC2_IP}")
print(f"S3 Bucket    : {S3_BUCKET}")
print("=" * 55)

EC2 COMMANDS — Copy and run these in EC2 browser terminal

# ── Step 1: Update system ─────────────────────────
sudo yum update -y

# ── Step 2: Install pip ───────────────────────────
sudo yum install python3-pip -y

# ── Step 3: Install Python libraries ─────────────
pip3 install pandas pyarrow boto3 matplotlib seaborn

# ── Step 4: Create project folder ────────────────
mkdir -p ~/weather_energy_pipeline/data/gold

# ── Step 5: Download data from S3 to EC2 ─────────
aws s3 sync s3://weather-energy-b5data/gold/ ~/weather_energy_pipeline/data/gold/ --region eu-north-1

# ── Step 6: Run Weather & Energy Pipeline Analysis ──
python3 -c "
import pyarrow.parquet as pq, pandas as pd, os, time
start = time.time()
# Find all parquet files in the gold directory
files = [os.path.join(r, f) for r, d, fs in os.walk(os.path.expanduser('~/weather_energy_pipeline/data/gold/')) for f in fs if f.endswith('.parquet')]
# Load and combine data
df = pd.concat([pq.read_table(f).to_pandas() for f in files]

In [38]:
print("=" * 55)
print("AWS DEPLOYMENT SUMMARY")
print("=" * 55)
print(f" S3 Bucket     : {S3_BUCKET}")
print(f" S3 Region     : {AWS_REGION} (Stockholm)")
print(f" Files uploaded : 220 Parquet files")
print(f" Upload size   : 407.5 MB")
print(f" EC2 Instance  : {EC2_INSTANCE_ID}")
print(f" EC2 IP        : {EC2_IP}")
print(f" EC2 Type      : t3.micro (1 vCPU, 1GB RAM)")
print(f" EC2 OS        : Amazon Linux 2023")
print(f" IAM Role      : {IAM_ROLE}")
print(f" Total Cost    : $0.00 (Free Tier)")
print("=" * 55)
print("Cloud deployment complete!")
print("=" * 55)

AWS DEPLOYMENT SUMMARY
 S3 Bucket     : weather-energy-b5data
 S3 Region     : eu-north-1 (Stockholm)
 Files uploaded : 220 Parquet files
 Upload size   : 407.5 MB
 EC2 Instance  : i-0704c4b2195d8e645
 EC2 IP        : 13.51.204.144
 EC2 Type      : t3.micro (1 vCPU, 1GB RAM)
 EC2 OS        : Amazon Linux 2023
 IAM Role      : ec2-s3-weather-role
 Total Cost    : $0.00 (Free Tier)
Cloud deployment complete!
